# als-recommender — walkthrough

A short end-to-end tour of the pure-numpy recommender core. We

1. import the package and run the seeded synthetic **demo**,
2. compare **biased** matrix factorisation against the plain (unbiased) model on
   bias-dominated data, and
3. compute the extra ranking metrics — **MAP** (via average precision),
   **MRR**, and **catalogue coverage**.

Everything here is pure numpy / pandas + stdlib: no PySpark, no JVM, runs in
seconds. The numbers are reproducible because every seed is fixed.


In [ ]:
import numpy as np
import recsys
from recsys import (
    als_factorize, predict,
    als_factorize_biased, predict_biased,
    als_implicit,
    average_precision_at_k, mean_reciprocal_rank, catalog_coverage,
)
from recsys.demo import run_demo

print("recsys", recsys.__version__)


## 1. The seeded demo

`run_demo(0)` drives the real core (ALS, popularity baseline, ranking metrics)
on small seeded synthetic low-rank data and returns both models' metrics plus
the ALS lift over popularity. On genuinely low-rank data, personalised ALS beats
the non-personalised baseline on every metric.


In [ ]:
result = run_demo(0)
print("ALS:        ", result["als"])
print("Popularity: ", result["popularity"])
print("Lift:       ", result["lift"])


## 2. Biased vs plain ALS

Some ratings matrices are dominated by *additive* effects: a user who rates
everything high, an item everyone loves. We build a **purely additive** matrix
`R[i,j] = mu + bu[i] + bv[j]` with no interaction structure, then fit both
models at the same rank. The biased model factors the offsets out explicitly
(`mu + bu + bv + U·Vᵀ`), so it fits the additive surface far better than the
plain factorisation — lower RMSE.


In [ ]:
# Purely additive matrix: R[i,j] = mu + bu[i] + bv[j], no interaction structure.
mu_true = 3.0
bu_true = np.array([-1.0, 0.0, 1.0, 2.0])
bv_true = np.array([0.5, -0.5, 1.0])
R = mu_true + bu_true[:, None] + bv_true[None, :]
mask = np.ones_like(R, dtype=bool)

def rmse(a, b):
    return float(np.sqrt(np.mean((a.ravel() - b.ravel()) ** 2)))

# Plain (unbiased) ALS at rank 1.
U0, V0 = als_factorize(R, mask, rank=1, reg=0.0, iters=40, seed=0)
plain_rmse = rmse(R, predict(U0, V0))

# Biased ALS at the same rank.
mu, bu, bv, U, V = als_factorize_biased(R, mask, rank=1, reg=0.0, iters=40, seed=0)
biased_rmse = rmse(R, predict_biased(mu, bu, bv, U, V))

print(f"plain  ALS rank-1 RMSE: {plain_rmse:.6f}")
print(f"biased ALS rank-1 RMSE: {biased_rmse:.6f}")
print("biased wins:", biased_rmse < plain_rmse)


In [ ]:
# Implicit feedback: two user groups liking disjoint item blocks.
P = np.array([
    [1.0, 1.0, 0.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.0],
    [0.0, 0.0, 1.0, 1.0],
])
C = 1.0 + 39.0 * P  # confidence = 1 + alpha * preference
Ui, Vi = als_implicit(C, P, rank=2, reg=0.1, iters=30, seed=0)
scores = predict(Ui, Vi)
top = scores.argmax(axis=1)
print("top item per user:", top.tolist(), "(each lands in its own block)")


## 3. Extra ranking metrics

Beyond Precision@K / Recall@K / NDCG@K, the metrics layer now offers:

- **average_precision_at_k** — the per-user building block of MAP.
- **mean_reciprocal_rank** — rewards getting one good item to the very top.
- **catalog_coverage** — what fraction of the catalogue is ever recommended (a
  diversity / aggregate-fairness check accuracy metrics miss).


In [ ]:
recommended = ["a", "b", "c", "d", "e"]
relevant = {"b", "d", "x"}
print("AP@4 :", average_precision_at_k(recommended, relevant, 4))   # 1/3

lists = [["a", "b", "c"], ["x", "y", "z"]]
rels  = [{"b"}, {"x"}]
print("MRR  :", mean_reciprocal_rank(lists, rels))                  # 0.75

catalog = {"a", "b", "c", "d"}
print("cover:", catalog_coverage([["a", "b"], ["b", "c"]], catalog))  # 0.75


## Takeaways

- ALS beats the popularity baseline on this low-rank data (offline lift, not
  online value — see the README caveats).
- The **biased** model wins on bias-dominated data because it represents
  additive user/item offsets directly.
- MAP / MRR / coverage round out the evaluation: ranking quality *and* how much
  of the catalogue actually gets surfaced.
